# USD Curve Parity Trace Demo

This notebook traces the USD discounting curve selection for a concrete ORE run and shows three things:

1. Which USD curve ORE selected from `todaysmarket.xml`
2. Which segment quotes and conventions feed `USD3M`
3. Which pillar outputs ORE produced in `todaysmarketcalibration.csv` and `curves.csv`


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.name and ROOT.name != "Tools":
    ROOT = ROOT.parent
if ROOT.name != "Tools":
    raise RuntimeError("Run this notebook from somewhere under the Engine workspace")

PYRUNNER = ROOT / "PythonOreRunner"
if str(PYRUNNER) not in sys.path:
    sys.path.insert(0, str(PYRUNNER))

from ore_curve_fit_parity.curve_trace import trace_usd_curve_from_ore

ORE_XML = PYRUNNER / "parity_artifacts" / "multiccy_benchmark_final" / "cases" / "flat_USD_5Y_B" / "Input" / "ore.xml"
TRACE_JSON = PYRUNNER / "parity_artifacts" / "usd_curve_trace" / "usd3m_flat_usd_5y_b.json"


In [ ]:
trace = trace_usd_curve_from_ore(ORE_XML)
TRACE_JSON.parent.mkdir(parents=True, exist_ok=True)
TRACE_JSON.write_text(json.dumps(trace, indent=2, sort_keys=True) + "\n", encoding="utf-8")

summary = {
    "configuration_id": trace["configuration_id"],
    "curve_handle": trace["curve_handle"],
    "curve_name": trace["curve_name"],
    "curve_id": trace["curve_config"]["curve_id"],
    "discount_curve_dependency": trace["curve_config"]["discount_curve"],
    "segment_count": len(trace["curve_config"]["segments"]),
    "ore_pillar_count": len(trace["ore_calibration_trace"]["pillars"]),
}
pd.Series(summary)


In [ ]:
segment_rows = []
for segment in trace["segment_alignment"]:
    convention = trace["conventions"].get(segment["conventions"], {})
    for quote in segment["quotes"]:
        ore_pillar = quote.get("ore_pillar") or {}
        segment_rows.append(
            {
                "segment_type": segment["type"],
                "convention_id": segment["conventions"],
                "convention_type": convention.get("convention_type", ""),
                "quote_key": quote["quote_key"],
                "market_quote": quote["quote_value"],
                "pillar_date": ore_pillar.get("date", ""),
                "pillar_time": float(ore_pillar.get("time", 0.0) or 0.0),
                "pillar_df": float(ore_pillar.get("discountFactor", 0.0) or 0.0),
                "pillar_zero": float(ore_pillar.get("zeroRate", 0.0) or 0.0),
            }
        )

segment_df = pd.DataFrame(segment_rows)
segment_df.head(12)


In [ ]:
curve_df = pd.DataFrame(
    {
        "calendar_date": trace["ore_curve_points"]["calendar_dates"],
        "time": trace["ore_curve_points"]["times"],
        "df": trace["ore_curve_points"]["dfs"],
    }
)
curve_df.head()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(curve_df["time"], curve_df["df"], label=trace["curve_name"], lw=2)
ax.scatter(segment_df["pillar_time"], segment_df["pillar_df"], color="black", s=18, label="ORE calibration pillars")
ax.set_title("USD3M ORE curve points and calibration pillars")
ax.set_xlabel("Time (years)")
ax.set_ylabel("Discount factor")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()
